### ETL

In [ ]:
import unicodedata
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType
import pycountry
import pycountry_convert
from rapidfuzz import fuzz, process


spark = (
    SparkSession.builder.appName("ETL_Full_Pipeline_BigTech")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2")
    .getOrCreate()
)

# =========================================================================
# 2. EXTRACCIÓN (E)
# =========================================================================
JDBC_URL = "jdbc:postgresql://seal-users.c180so2u4aci.us-east-2.rds.amazonaws.com:5432/Users"
PROPIEDADES = {
    "user": "postgres",
    "password": "*******",
    "driver": "org.postgresql.Driver",
}

subquery = '(SELECT cognito_id, email, username, birthdate, country, state, created_at FROM "usuarios") as temp_usuarios'

df_spark = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("dbtable", subquery)
    .options(**PROPIEDADES)
    .load()
)

# =========================================================================
# 3. TRANSFORMACIÓN (T) - LIMPIEZA Y EDAD VECTORIZADA
# =========================================================================
# Reemplazo de .str.lower().str.strip() por funciones nativas distribuidas
df_cleaned = (
    df_spark.withColumn("cognito_id", F.lower(F.trim(F.col("cognito_id"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("username", F.lower(F.trim(F.col("username"))))
    .withColumn("country", F.lower(F.trim(F.col("country"))))
    .withColumn("state", F.lower(F.trim(F.col("state"))))
    # Casteo estricto a Tipo Fecha
    .withColumn("birthdate", F.to_date(F.trim(F.col("birthdate")), "yyyy-MM-dd"))
)

# Cálculo de edad vectorizado (sin bisiestos rotos ni loops)
df_transformado = df_cleaned.withColumn(
    "edad",
    F.floor(F.months_between(F.current_date(), F.col("birthdate")) / 12).cast(
        IntegerType()
    ),
).withColumn(
    "rango_edad",
    F.when((F.col("edad") >= 0) & (F.col("edad") <= 12), "Niño")
    .when((F.col("edad") >= 13) & (F.col("edad") <= 17), "Adolescente")
    .when((F.col("edad") >= 18) & (F.col("edad") <= 24), "Joven")
    .when((F.col("edad") >= 25) & (F.col("edad") <= 34), "Adulto Joven")
    .when((F.col("edad") >= 35) & (F.col("edad") <= 44), "Adulto")
    .when((F.col("edad") >= 45) & (F.col("edad") <= 54), "Adulto Maduro")
    .when((F.col("edad") >= 55) & (F.col("edad") <= 64), "Adulto Mayor")
    .when(F.col("edad") > 64, "Tercera Edad")
    .otherwise("Desconocido"),
)

# Reemplazo de np.random.choice en Spark distribuido
# Usamos un generador pseudoaleatorio basado en una semilla hash
df_transformado = df_transformado.withColumn(
    "membresia",
    F.when(F.rand(seed=42) > 0.5, "premium").otherwise("normal"),
)

# =========================================================================
# 4. PREPARACIÓN DE CATÁLOGOS PARA LAS UDFs
# =========================================================================
def normalizar(texto):
    texto = str(texto).strip().lower()
    return "".join(
        c
        for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


# Inicialización de diccionarios que viajarán a los nodos del clúster
paises_norm = {normalizar(c.name): c.name for c in pycountry.countries}

estados_por_iso = {}
for sub in pycountry.subdivisions:
    estados_por_iso.setdefault(sub.country_code, []).append(sub.name)

estados_norm = {
    iso: {normalizar(e): e for e in estados}
    for iso, estados in estados_por_iso.items()
}

continent_map = {
    "AF": "África",
    "AS": "Asia",
    "EU": "Europa",
    "NA": "América del Norte",
    "SA": "América del Sur",
    "OC": "Oceanía",
    "AN": "Antártida",
}


# =========================================================================
# 5. CREACIÓN DE USER DEFINED FUNCTIONS (UDFs) PARA VALIDACIÓN DIFUSA
# =========================================================================
@F.udf(returnType=StringType())
def udf_corregir_pais(pais):
    if not pais:
        return "Desconocido"
    pais_norm = normalizar(pais)
    match = process.extractOne(
        pais_norm, paises_norm.keys(), scorer=fuzz.WRatio
    )
    if match and match[1] >= 80:
        return paises_norm[match[0]]
    return pais.capitalize()


@F.udf(returnType=StringType())
def udf_corregir_estado(pais, estado):
    if not estado or not pais:
        return "Desconocido"

    pais_obj = pycountry.countries.get(name=pais)
    if not pais_obj:
        # Intento de rescate si el nombre difiere un poco
        matches = pycountry.countries.search_fuzzy(pais)
        if matches:
            pais_obj = matches[0]
        else:
            return "Desconocido"

    iso = pais_obj.alpha_2
    catalogo = estados_norm.get(iso)
    if not catalogo:
        return "Desconocido"

    estado_norm = normalizar(estado)
    match = process.extractOne(estado_norm, catalogo.keys(), scorer=fuzz.WRatio)

    if match and match[1] >= 10:  # Mantenemos tu umbral del 10%
        return catalogo[match[0]]
    return "Desconocido"


@F.udf(returnType=StringType())
def udf_obtener_continente(pais):
    if not pais or pais == "Desconocido":
        return "Desconocido"
    try:
        country_matches = pycountry.countries.search_fuzzy(str(pais).strip())
        if not country_matches:
            return "Desconocido"

        country_obj = country_matches[0]
        continent_code = pycountry_convert.country_alpha2_to_continent_code(
            country_obj.alpha_2
        )
        return continent_map.get(continent_code, "Desconocido")
    except:
        return "Desconocido"


# =========================================================================
# 6. EJECUCIÓN ORQUESTADA DE LAS UDFs
# =========================================================================
# 1. Corregimos el país
df_final = df_transformado.withColumn(
    "country", udf_corregir_pais(F.col("country"))
)

# 2. Corregimos el estado basándonos en el nuevo país resuelto
df_final = df_final.withColumn(
    "state", udf_corregir_estado(F.col("country"), F.col("state"))
)

# 3. Asignamos el continente correspondiente
df_final = df_final.withColumn(
    "continent", udf_obtener_continente(F.col("country"))
)

# =========================================================================
# 7. SALIDA DE DATOS
# =========================================================================
df_final.show(5, truncate=False)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.